In [1]:
import os
import heapq
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from pathlib import Path
import heapq
import re

In [2]:

df = pd.read_csv(r"F:\sanskrit NLP\Works\notebooks\inputs\sanskrit_phoneme_vectors (1).csv", index_col=0)

# Access the values of each row by slicing the tuple from index 1
phoneme_vectors = {row.Index: np.array(row[1:], dtype=float) for row in df.itertuples()}

print(f"Loaded {len(phoneme_vectors)} Sanskrit phonemes with 34 features.")
print(phoneme_vectors)


Loaded 52 Sanskrit phonemes with 34 features.
{'अ': array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -1.,  1., -1.,  1.,
       -1., -1.,  1.,  0.,  0., -1., -1., -1.]), 'आ': array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -1.,  1., -1.,  1.,
       -1., -1.,  1.,  0.,  0., -1.,  1., -1.]), 'इ': array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -1., -1., -1., -1.,
        1., -1., -1.,  0.,  0., -1., -1., -1.]), 'ई': array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -1., -1., -1., -1.,
        1., -1., -1.,  0.,  0., -1.,  1., -1.]), 'उ': array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -1., -1., -1., -1.,
       -1., 

In [3]:

def load_sanskrit_words(path):
    words = []
    with open(path, "r", encoding="utf-8") as f:
        for text in f:
            text = re.sub(r"[०-९\d]", "", text)        # remove digits
            text = re.sub(r"[।॥]", " ", text)         # replace danda marks with space
            text = re.sub(r"[^ऀ-ॿ ]", " ", text)      # keep only Devanagari + space
            text = re.sub(r"\s+", " ", text)
            for w in text.split():
                if w:
                    words.append(w)
    return words

corpus_path = r"F:\sanskrit NLP\Works\notebooks\inputs\161656_RV_Terms.txt"
words = load_sanskrit_words(corpus_path)
print(len(words))
print("Example:", words[:100])
vocab = set(words)
print("unique words: ",len(vocab))

183422
Example: ['असा॑वि', 'सोमः', 'इन्द्र', 'ते॒', 'शवि॑ष्ठ', 'धृष्णो', 'इति', 'आ', 'गहि', 'आ', 'त्वा', 'पृणक्तु', 'इन्द्रियम्', 'रजः॑', 'सूर्यः॑', 'न', 'र', 'इन्द्रम्', 'इत्', 'हरी॒', 'इति॑', 'व॒ह॒तः॒', 'अप्र॑तिधृष्टऽशवसम्', 'ऋषीणाम्', 'च', 'स्तुतीः', 'उप॑', 'यज्ञम्', 'च', 'मानु॑षाणाम्', 'आ', 'तिष्ठ', 'वृत्र', 'हन्', 'रथम्', 'यु॒क्ता', 'ते', 'ब्रह्मणा', 'हरी॒', 'इति॑', 'अर्वाचीनम्', 'सु', 'ते॒', 'मनः', 'ग्रावा', 'कृणोतु', 'व॒ग्नुना॑', 'इमम्', 'इन्द्र', 'सुतम्', 'पिब', 'ज्येष्ठम्', 'अमर्त्यम्', 'मदम्', 'शुक्रस्य', 'त्वा', 'अभि', 'अक्षरन्', 'धाराः', 'ऋतस्य', 'सद॑ने', 'इन्द्राय', 'नूनम्', 'अर्चत', 'उक्थानि', 'च', 'ब्रवीतन', 'सुताः', 'अमत्सुः', 'इन्दवः', 'ज्येष्ठम्', 'नमस्यत', 'स', 'नकिः', 'त्वत्', 'र॒थिऽत॑रः', 'हरी॒', 'इति॑', 'यत्', 'इन्द्र', 'यच्छसे', 'नकिः', 'त्वा', 'अनु', 'मज्मना', 'नकिः', 'सु', 'अश्वः', 'आ', 'यः', 'एकः॑', 'इत्', 'वि॒ऽदय॑ते', 'वसु॑', 'मर्ताय', 'दा॒शुषे॑', 'ईशा॑नः', 'अप्र॑तिऽस्कुतः', 'इन्द्रः॑', 'अ॒ङ्ग']
unique words:  31064


In [4]:


def panphon_segment_distance(a, b):
    if a not in phoneme_vectors or b not in phoneme_vectors:
        return 1.0  # Max distance if unknown phoneme
    vec_a = phoneme_vectors[a]
    vec_b = phoneme_vectors[b]
    return np.sum(np.abs(vec_a - vec_b)) / len(vec_a) # normalized by 34 (features)


In [5]:
def panphon_word_distance(word1: str, word2: str) -> float:
    n, m = len(word1), len(word2)
    dp = np.zeros((n + 1, m + 1))

    for i in range(1, n + 1): # insertion cost
        dp[i][0] = i
    for j in range(1, m + 1): # deletion cost
        dp[0][j] = j

    # Compute DP table
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub_cost = panphon_segment_distance(word1[i - 1], word2[j - 1])
            dp[i][j] = min(
                dp[i - 1][j] + 1,  # deletion
                dp[i][j - 1] + 1,  # insertion
                dp[i - 1][j - 1] + sub_cost  # substitution
            )

    # Normalize by max word length
    return dp[n][m] / max(n, m)

In [8]:
panphon_word_dist = panphon_word_distance


In [9]:
def length_prune(w1, w2, max_diff=2):
    return abs(len(w1) - len(w2)) <= max_diff


In [10]:
def mine_triplet_for_anchor(anchor, vocab):
    min_d = float("inf")
    max_d = -1.0

    positive = None
    negative = None

    for other in vocab:
        if other == anchor:
            continue

        # OPTIONAL pruning (comment out if you want full search)
        if not length_prune(anchor, other):
            continue

        d = panphon_word_dist(anchor, other)

        if d < min_d:
            min_d = d
            positive = other

        if d > max_d:
            max_d = d
            negative = other

    return {
        "anchor": anchor,
        "positive": positive,
        "negative": negative,
        "min_distance": min_d,
        "max_distance": max_d
    }


In [11]:
from joblib import Parallel, delayed
from tqdm import tqdm
import multiprocessing as mp

n_jobs = mp.cpu_count()


In [ ]:
triplets = Parallel(
    n_jobs=n_jobs,
    backend="loky",
    batch_size=1  # keep memory stable
)(
    delayed(mine_triplet_for_anchor)(anchor, vocab)
    for anchor in tqdm(vocab, desc="Mining triplets")
)


Mining triplets:   5%|▍         | 1408/31064 [47:14<19:30:48,  2.37s/it]

In [ ]:
import pandas as pd

triplet_df = pd.DataFrame(triplets)
triplet_df.head()
triplet_df.to_csv("panphon_triplets.csv", index=False, encoding = 'utf-8-sig')
